# Failure explorer

Browse every grounding failure, **choose which ones to render yourself**, and export each
chosen case as a 3-D scene you can open in a real point-cloud viewer.

| box | colour |
|---|---|
| ground truth | green |
| prediction | red |
| other objects of the same class | grey |

All logic is imported from `experiments/analysis.py`; nothing is reimplemented here.

In [ ]:
# ---- config --------------------------------------------------------------------------
PREDICTIONS   = "outputs/2024-12-18_20-40-38_3DVG-FIXED/predictions.p"
OUT_ROOT      = "vis_failure_scenes"
SPLIT         = "val"

SOURCE        = "ply"        # "ply": the original ~160k scans. "npy": the detector's 50k.
SCANS_ROOT    = None         # None uses data/scannet/scans
DPI           = 200
MAX_POINTS    = 200000       # cap for the PNG only; scene.ply is never subsampled
BOX_SPACING   = 0.005        # gap between the points making up a cage edge
WRITE_FIGURE  = True         # also render the usual three-panel PNG per case

# ---- which failures to list ----------------------------------------------------------
CAUSES        = []           # e.g. ["small_object", "distractor_confusion"]; [] = all
SCENES        = []           # e.g. ["scene0011_00"]; [] = all
IOU_MIN       = None
IOU_MAX       = None
PICK_LIMIT    = 0           # rows listed per cause; 0 lists every one

# Leave empty to be prompted. Set it to re-run a known set without a prompt:
#   SELECTION = "1-3,17"
SELECTION     = ""

# ---- failure definition (matches `failure-cases` defaults) ---------------------------
THRESHOLD        = 0.25      # a failure is IoU below this
CRITERION        = "fuzzy"
PARSE_FOLDER     = "final_parsing_tokenized"
SMALL_PERCENTILE = 10.0
LONG_PERCENTILE  = 75.0
MANY_SPATIAL     = 3
DISTRACTOR_IOU   = 0.05
DRIFT_LOW        = 0.10

In [ ]:
import os, sys, json, pickle
from argparse import Namespace
from collections import Counter

import numpy as np

if not os.path.isfile("experiments/analysis.py"):       # be runnable from a subdirectory
    os.chdir(os.path.dirname(os.path.abspath("")) or ".")
sys.path.insert(0, os.getcwd())

from experiments.analysis import (
    load_scanrefer, load_predictions, join, load_parse_cache, parse_for,
    scene_class_counts, attribute, spatial_hits, box_volume, match_kinds,
    filter_failures, format_failure_listing, parse_selection, pick_cases,
    export_scene, md_table, CAUSES as ALL_CAUSES, CAUSE_TEXT,
)

print(f"cwd {os.getcwd()}")

In [ ]:
# ---- build the failure table ---------------------------------------------------------
# The same computation `failure-cases` runs, kept in memory so the picker can work on it.
ARGS = Namespace(criterion=CRITERION, distractor_iou=DISTRACTOR_IOU,
                 drift_low=DRIFT_LOW, many_spatial=MANY_SPATIAL)

records = load_scanrefer(SPLIT)
rows, missing = join(load_predictions(PREDICTIONS), records)
print(f"{len(rows)} annotations joined" + (f"  ({missing} without a prediction)" if missing else ""))

try:
    cache = load_parse_cache(PARSE_FOLDER, SPLIT)
except FileNotFoundError as error:
    print(f"WARNING: {error}\n         parse-based causes will be unavailable")
    cache = {}

counts = scene_class_counts(records)
small_threshold = float(np.percentile(
    [box_volume(r["_prediction"]["gt_bbox"]) for r in rows], SMALL_PERCENTILE))
long_threshold = float(np.percentile(
    [len(r.get("token") or []) for r in rows], LONG_PERCENTILE))

failures = []
for row in rows:
    if row["iou"] >= THRESHOLD:
        continue
    parsed = parse_for(cache, row["scene_id"], row["object_id"], row["ann_id"]) or {}
    predicted_target = " ".join(parsed.get("target") or [])
    target_ok = (match_kinds(predicted_target, row["object_name"])[CRITERION]
                 if parsed else True)
    signals = {
        "parse_target_ok": bool(target_ok),
        "same_class_instances": counts[row["scene_id"]].get(row["object_name"], 1),
        "gt_volume": round(box_volume(row["_prediction"]["gt_bbox"]), 4),
        "small_threshold": round(small_threshold, 4),
        "num_tokens": int(len(row.get("token") or [])),
        "long_threshold": int(long_threshold),
        "num_spatial": len(spatial_hits(row.get("token") or [])),
    }
    failures.append({
        "scene_id": row["scene_id"], "object_id": row["object_id"],
        "ann_id": row["ann_id"], "object_name": row["object_name"],
        "description": row.get("description", ""),
        "iou": row["iou"], "cause": attribute(row, signals, ARGS), "signals": signals,
        "parsed_target": predicted_target or "(no parse)",
        "parsed_adjectives": " ".join(parsed.get("adjectives") or []) or "(no parse)",
        "parsed_neighbors": " ".join(parsed.get("neighbors") or []) or "(no parse)",
    })

tally = Counter(c["cause"] for c in failures)
print(f"\n{len(failures)} of {len(rows)} annotations "
      f"({100.0 * len(failures) / max(len(rows), 1):.2f}%) fall below IoU {THRESHOLD}\n")
print(md_table(["cause", "count", "% of failures"],
               [[c, tally[c], f"{100.0 * tally[c] / max(len(failures), 1):.1f}"]
                for c in ALL_CAUSES if tally[c]]))

In [ ]:
# ---- list the candidates and pick ----------------------------------------------------
# Grouped by cause, each group worst-IoU first. Indices run globally, so "3,17-20"
# addresses any row. Set SELECTION in the config cell to skip the prompt.
pool = filter_failures(failures, CAUSES, SCENES, IOU_MIN, IOU_MAX)
if len(pool) != len(failures):
    print(f"[filter] {len(pool)} of {len(failures)} failures match\n")

chosen = pick_cases(pool, PICK_LIMIT, SELECTION if SELECTION else None) if pool else []
print(f"\n{len(chosen)} case(s) chosen")
for rank, case in enumerate(chosen, 1):
    print(f"  [{rank}] {case['cause']:<22} IoU={case['iou']:.3f}  {case['scene_id']} "
          f"obj{case['object_id']} ({case['object_name']})")

In [ ]:
# ---- export -------------------------------------------------------------------------
EXPORT_ARGS = Namespace(source=SOURCE, scans_root=SCANS_ROOT, box_spacing=BOX_SPACING,
                        write_figure=WRITE_FIGURE, dpi=DPI, max_points=MAX_POINTS)

with open(PREDICTIONS, "rb") as handle:
    predictions = pickle.load(handle)
os.makedirs(OUT_ROOT, exist_ok=True)

exported = []
for rank, case in enumerate(chosen, 1):
    print(f"[{rank}/{len(chosen)}] {case['scene_id']}/{case['object_id']}/{case['ann_id']} "
          f"[{case['cause']}]")
    result = export_scene(case, predictions, OUT_ROOT, EXPORT_ARGS)
    if result:
        result["case"] = case
        exported.append(result)
        flag = "  FRAME MISMATCH" if result["misaligned"] else ""
        print(f"      {result['num_points']:,} points ({result['source']})  "
              f"{result['bytes'] / 1e6:.1f} MB{flag}")
        print(f"      {os.path.abspath(result['scene'])}")

print(f"\nwrote {len(exported)} scene(s). Open a scene.ply in MeshLab or CloudCompare:")
print("green = ground truth, red = prediction, grey = other objects of the same class.")

In [ ]:
# ---- what was written, and the figures inline -----------------------------------------
from IPython.display import Image, display

if exported:
    print(md_table(
        ["cause", "scene", "IoU", "points", "MB", "distractors", "frame offset (m)"],
        [[i["case"]["cause"], i["case"]["scene_id"], f"{i['iou']:.3f}",
          f"{i['num_points']:,}", f"{i['bytes'] / 1e6:.1f}", i["distractors"],
          "-" if i["frame_offset"] is None else f"{i['frame_offset']:.4f}"]
         for i in exported]))

for item in exported:
    if item["figure"]:
        display(Image(filename=item["figure"]))